# Aim
The aim of this notebook is to test the analysis pipeline for Thomas's data on a subset of images and compare to manual quantification.



# 1. Functions and imports

In [3]:
from imaris_ims_file_reader.ims import ims
import numpy as np
import os
import re
from tqdm import tqdm
import pyclesperanto_prototype as cle
import matplotlib.pyplot as plt
import napari
from skimage.morphology import skeletonize, medial_axis
from skan.csr import skeleton_to_csgraph, Skeleton, summarize
from skan import draw
from skimage.feature import shape_index
from skimage.measure import regionprops
from scipy.spatial import cKDTree
import pandas as pd
from stackview import insight
from apoc import ObjectSegmenter
import seaborn as sns
from scipy.ndimage import distance_transform_edt, center_of_mass
from skimage.segmentation import find_boundaries
from skimage.filters import meijering


In [4]:
import limoncello as lc

In [3]:
import contextlib
from functools import wraps

def suppress_print_keep_tqdm(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        with open(os.devnull, 'w') as f, contextlib.redirect_stdout(f):
            return func(*args, **kwargs)
    return wrapper

In [6]:
def cilia_process_3d_ml(volume,classifier_path="c:/Users/qfavey/Documents/Code-Thomas-Test/LimonCELLo/Segmenters/cilia-segmenter.cl",min_size=20):
    segmenter = ObjectSegmenter(opencl_filename=classifier_path)
    result = segmenter.predict(image=volume)
    filtered_result = cle.exclude_small_labels(cle.push(result), maximum_size=min_size)
    return filtered_result

In [12]:
def nuclei_process_3d(
    volume,
    tophat_radius=(2.0, 2.0, 2.0),
    dilation_radius=(2.0, 2.0, 2.0),
    min_size=50,
    spot_sigma = 5, 
    outline_sigma = 1
):
    """
    3D cilia segmentation pipeline on GPU.

    Parameters
    ----------
    volume : np.ndarray
        3D array (Z, Y, X)
    tophat_radius : tuple
        (rx, ry, rz) for 3D top-hat sphere
    dilation_radius : tuple
        (rx, ry, rz) for 3D label dilation
    min_size : int
        Minimum voxel size to keep objects
    spot_sigma: int
        How close detected cells can be
    outline_sigma: int
        How precise the segmented objects are outlined

    Returns
    -------
    result : np.ndarray
        3D labeled volume
    """

    # Push to GPU
    volume_gpu = cle.push(volume)

    # 3D top-hat (sphere)
    th_gpu = cle.top_hat_sphere(
        volume_gpu,
        radius_x=tophat_radius[2],
        radius_y=tophat_radius[2],
        radius_z=tophat_radius[2],
    )

    # 3D Otsu threshold
    labels_gpu = cle.voronoi_otsu_labeling(th_gpu,spot_sigma=spot_sigma, outline_sigma=outline_sigma)

    # Pull back to CPU
    result = cle.pull(labels_gpu)

    return result


def neurites_process_gpu_filter(
    neurites,
    nuclei_labels,
    min_size=50,
    spot_sigma=10,
    outline_sigma=1,
    meijering_sigmas=(1, 2, 3),  
):

    # ---- Push to GPU ----
    neurites_gpu = cle.push(neurites)

    # ---- Segmentation (Voronoi Otsu on enhanced image) ----
    neurites_label = cle.exclude_small_labels(
        cle.voronoi_otsu_labeling(
            neurites_gpu,
            spot_sigma=spot_sigma,
            outline_sigma=outline_sigma
        ),
        maximum_size=min_size
    )

    # ---- Skeletonize (CPU fallback) ----
    binary = cle.pull(neurites_label) > 0
    skeleton = skeletonize(binary)

    return skeleton, neurites_label

def neurites_process_gpu(neurites, min_size=50, spot_sigma=10,outline_sigma=1):
    """
     3D cilia segmentation pipeline on GPU.

    Parameters
    ----------
    neurites : np.ndarray
        3D array (Z, Y, X)
    min_size : int
        Minimum voxel size to keep objects
    spot_sigma: int
        How close detected cells can be
    outline_sigma: int
        How precise the segmented objects are outlined

    Returns
    -------
    skeleton: np.ndarray
        3D skeleton
    result : np.ndarray
        3D labeled volume
    """
    # ---- Push to GPU ----
    neurites_gpu = cle.push(neurites)
    

    # ---- Tubular enhancement (DoG instead of Frangi) ----
    # blurred_small = cle.gaussian_blur(neurites_gpu, 1, 1, 0)
    # blurred_large = cle.gaussian_blur(neurites_gpu, 3, 3, 0)
    # enhanced_gpu = cle.subtract_images(blurred_small, blurred_large)

    # ---- Threshold (GPU Otsu) ----
    neurites_label =  cle.exclude_small_labels(cle.voronoi_otsu_labeling(neurites_gpu, spot_sigma=spot_sigma, outline_sigma=outline_sigma),maximum_size=min_size)


    # ---- Skeletonize (CPU fallback) ----
    binary = cle.pull(neurites_label)

    skeleton = skeletonize(binary)
    #skel_med, dist = medial_axis(binary, return_distance=True)

    # ---- GPU Voronoi labeling instead of EDT ----
    # This assigns each pixel to nearest label directly on GPU


    return skeleton, neurites_label

In [5]:

ims_path = "H:/tlobri/2. Experiments, Differentiations and Adaptations/4. Cilia on Neurites/EXP_E6_250807/250819 - BC43 IF WTSII ROIs/EXP_E6_d40_WTSII_DAPI_PCN488_TUJ1568_ARL13B647_40x_2025-08-19_(9).ims"

print(cle.available_device_names(dev_type="gpu"))
cle.select_device('NVIDIA GeForce RTX 4090')  # optional but good practice
print(cle.get_device())

['NVIDIA GeForce RTX 4090', 'gfx1036']
<NVIDIA GeForce RTX 4090 on Platform: NVIDIA CUDA (1 refs)>


In [ ]:
def run_pipeline3(
    input_path,
    output_path,
    max_cilia_dist_cutoff_um=2.0,
    nuclei_spot_sigma=15,
    tophat_radius=12,
    neurite_spot_sigma=5,
):
    os.makedirs(output_path, exist_ok=True)
    csv_dir = os.path.join(output_path, "csv")
    os.makedirs(csv_dir, exist_ok=True)

    all_dfs = []

    for file in tqdm(os.listdir(input_path)):
        if not file.endswith(".ims"):
            continue

        print(f"Processing {file}...")

        # Load data
        a, meta = lc.load_image(os.path.join(input_path, file))
        voxel_size = meta["voxel_size"]
        
        a_norm = lc.normalize_intensity(a,p_low=2,p_high=98)
        
        # Segmentation
        cilia_labels = lc.segment_cilia_ml(
            a[0, 0],
            classifier_path = r'C:\Users\qfavey\Documents\Code-Thomas-Test\LimonCELLo\segmenters\cilia-segmenter.cl',
            )

        nuclei_labels_otsu = lc.segment_nuclei(
            a_norm[0, 3],
            tophat_radius=(tophat_radius, tophat_radius, tophat_radius),
            spot_sigma=nuclei_spot_sigma,
            outline_sigma=3,
        )

        skeleton, neurites_label = lc.segment_neurites(
            a_norm[0, 1],
            spot_sigma=neurite_spot_sigma,
        )

        basal_bodies_labels = lc.segment_basal_bodies(
            a[0,2]
        )

        # get neurites masks
        neurite_mask = np.asarray(neurites_label) > 0
        skeleton_mask = np.asarray(skeleton) > 0

        # Nuclei distance map
        distance_map_nuclei = distance_transform_edt(
            ~(nuclei_labels_otsu > 0), sampling=voxel_size
        )

        # Neurite thickness proxy (radius)
        distance_map_neurites = distance_transform_edt(
            neurite_mask, sampling=voxel_size
        )

        # Ratio
        map_ratio = distance_map_nuclei / distance_map_neurites
        map_ratio[~np.isfinite(map_ratio)] = np.nan

        # Distance to neurites (for filtering)
        dist_to_neurite = distance_transform_edt(
            ~neurite_mask,
            sampling=voxel_size
        )

        # Nearest SKELETON mapping 
        _, nearest_skel_idx = distance_transform_edt(
            ~skeleton_mask,
            return_indices=True,
            sampling=voxel_size,
        )

        # Cilia centroids
        cilia_ids = np.unique(cilia_labels)
        cilia_ids = cilia_ids[cilia_ids != 0]

        centroids = center_of_mass(
            cilia_labels > 0, labels=cilia_labels, index=cilia_ids
        )

        rows = []

        for cid, centroid in zip(cilia_ids, centroids):

            z, y, x = np.round(centroid).astype(int)
            
            z = np.clip(z, 0, cilia_labels.shape[0] - 1)
            y = np.clip(y, 0, cilia_labels.shape[1] - 1)
            x = np.clip(x, 0, cilia_labels.shape[2] - 1)

            # d = dist_to_neurite[z, y, x]
            from scipy.ndimage import map_coordinates
            d = map_coordinates(dist_to_neurite, np.array(centroid).reshape(3,1), order=1)[0]

            if d > max_cilia_dist_cutoff_um:
                continue

            #  nearest skeleton voxel (NOT boundary anymore)
            sz, sy, sx = nearest_skel_idx[:, z, y, x]

            dt_neurite = distance_map_neurites[sz, sy, sx]
            dt_nuclei = distance_map_nuclei[sz, sy, sx]
            ratio = map_ratio[sz, sy, sx]

            rows.append(
                {
                    "filename": file,
                    "cilia_id": cid,
                    "centroid_z": centroid[0],
                    "centroid_y": centroid[1],
                    "centroid_x": centroid[2],
                    "distance_to_neurite_um": d,
                    "ratio": ratio,
                    "log_ratio": np.log(ratio),
                    "dt_neurite": dt_neurite,
                    "dt_nuclei": dt_nuclei,
                    "log_dt_neurite": np.log(dt_neurite),
                    "log_dt_nuclei": np.log(dt_nuclei),


                }
            )

        df = pd.DataFrame(rows)

        if df.empty:
            print(f"No valid cilia found in {file}")
            continue

        all_dfs.append(df)

        # Overlay visualization (MIP)
        overlay_dir = os.path.join(output_path, "figures", "overlays")
        os.makedirs(overlay_dir, exist_ok=True)

        # MIP of neurite + cilia channels
        neurite_mip = np.max(a_norm[0, 1], axis=0)   # (Y, X)
        cilia_mip = np.max(a_norm[0, 0], axis=0)     # (Y, X)
        nuclei_mip = np.max(a_norm[0,3], axis = 0)
       
        fig, axes = plt.subplots(2,2, figsize=(14,10))

        # Base layer: neurites
        axes[0,0].imshow(neurite_mip, cmap="gray")
        axes[0,0].set_title('Neurites MIP')
        axes[1,1].imshow(cilia_mip,cmap='gray')
        axes[1,1].set_title('Cilia MIP')
        axes[0,1].imshow( np.log(map_ratio[map_ratio.shape[0] // 2]),cmap='coolwarm')
        axes[0,1].set_title('log(ratio) overlay')
        axes[1,0].imshow(nuclei_mip,cmap='gray')
        axes[1,0].set_title('Nuclei MIP')

        # Normalize scores for coloring (robust)
        scores = df["log_ratio"].values
        valid_scores = scores[np.isfinite(scores)]

        if len(valid_scores) > 0:
            vmin, vmax = np.percentile(valid_scores, [5, 95])
        else:
            vmin, vmax = 0, 1

        norm = plt.Normalize(vmin=vmin, vmax=vmax)
        cmap = plt.cm.coolwarm

        # Overlay cilia centroids
        ys = df["centroid_y"].values
        xs = df["centroid_x"].values
        colors = cmap(norm(scores))

        axes[0,0].scatter(
            xs,
            ys,
            c=colors,
            s=10,
            edgecolor="black",
            linewidth=0.3,
            
        )
        axes[1,1].scatter(
            xs,
            ys,
            c=colors,
            s=10,
            edgecolor="black",
            linewidth=0.3,
            
        )
        axes[0,1].scatter(
            xs,
            ys,
            c=colors,
            s=10,
            edgecolor="black",
            linewidth=0.3,
            
        )
        axes[1,0].scatter(
            xs,
            ys,
            c=colors,
            s=10,
            edgecolor="black",
            linewidth=0.3,
            
        )

        # Colorbar
        sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
        sm.set_array([])
        plt.colorbar(sm, ax=axes[1,0], label="log_ratio")
        plt.colorbar(sm, ax=axes[1,1], label="log_ratio")
        plt.colorbar(sm, ax=axes[0,0], label="log_ratio")
        plt.colorbar(sm, ax=axes[0,1], label="log_ratio")

        plt.axis("off")

        save_path = os.path.join(
            overlay_dir,
            f"{os.path.splitext(file)[0]}_overlay.png"
        )

        plt.savefig(save_path, dpi=300, bbox_inches="tight")
        plt.close()

    if len(all_dfs) == 0:
        print("No data processed.")
        return

    final_df = pd.concat(all_dfs, ignore_index=True)


    # Transform
    final_df["log_ratio"] = np.log(final_df["ratio"])

    # Short names
    unique_files = final_df["filename"].unique()
    file_map = {f: f"S{i+1}" for i, f in enumerate(unique_files)}
    final_df["file_short"] = final_df["filename"].map(file_map)

    
    # Classification
    def classify(score):
        if score > 2.5:
            return "axon"
        elif score < 1:
            return "soma"
        else:
            return "ambiguous"

    final_df["class"] = final_df["log_ratio"].apply(classify)

    # Figures (clean + meaningful)
    fig_dir = os.path.join(output_path, "figures")
    os.makedirs(fig_dir, exist_ok=True)

    
    # 1. Multi-panel (ratio + log-ratio)
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    # Ratio
    sns.histplot(final_df["ratio"], kde=True, ax=axes[0, 0])
    axes[0, 0].set_title("Raw Ratio Distribution")

    # Log ratio
    sns.histplot(final_df["log_ratio"], kde=True, ax=axes[0, 1])
    axes[0, 1].set_title("Log Ratio Distribution")

    # Log ratio per sample (boxplot)
    sns.boxplot(data=final_df, x="file_short", y="log_ratio", ax=axes[1, 0])
    axes[1, 0].set_title("Log Ratio per Sample")
    axes[1, 0].tick_params(axis='x', rotation=45)

    # Class distribution per sample
    sns.countplot(data=final_df, x="file_short", hue="class", ax=axes[1, 1])
    axes[1, 1].set_title("Class Distribution per Sample")
    axes[1, 1].tick_params(axis='x', rotation=45)

    plt.tight_layout()
    plt.savefig(os.path.join(fig_dir, "overview_panels.png"), dpi=300)
    plt.close()

    # 2. KDE per sample
    plt.figure(figsize=(10, 6))

    sns.kdeplot(
        data=final_df,
        x="log_ratio",
        hue="file_short",
        common_norm=False,
    )

    plt.title("Log Ratio Distribution per Sample")
    plt.xlabel("Log Ratio")

    plt.savefig(os.path.join(fig_dir, "logratio_kde_per_sample.png"), dpi=300)
    plt.close()

    # QC + Excel
    excel_path = os.path.join(csv_dir, "all_cilia_features.xlsx")

    qc_sample = final_df.groupby("file_short").agg(
        n_cilia=("cilia_id", "count"),
        mean_ratio=("ratio", "mean"),
        std_ratio=("ratio", "std"),
        mean_log_ratio=("log_ratio", "mean"),
        std_log_ratio=("log_ratio", "std"),
        mean_distance=("distance_to_neurite_um", "mean"),
        max_distance=("distance_to_neurite_um", "max"),
    ).reset_index()

    qc_global = pd.DataFrame({
        "metric": [
            "n_total_cilia",
            "mean_ratio",
            "std_ratio",
            "mean_log_ratio",
            "std_log_ratio"
        ],
        "value": [
            len(final_df),
            final_df["ratio"].mean(),
            final_df["ratio"].std(),
            final_df["log_ratio"].mean(),
            final_df["log_ratio"].std(),
        ]
    })

    with pd.ExcelWriter(excel_path) as writer:
        final_df.to_excel(writer, sheet_name="all_data", index=False)
        qc_sample.to_excel(writer, sheet_name="qc_per_sample", index=False)
        qc_global.to_excel(writer, sheet_name="qc_global", index=False)

    print(f"Saved QC Excel: {excel_path}")


    # Scatter plot (the moment of truth)
    plt.figure(figsize=(10, 6))

    sns.scatterplot(
        x='dt_neurite',
        y='dt_nuclei',
        hue='file_short',
        data=final_df,
        palette='tab10'
    )

    plt.xlabel("Assigned Neurite Voxel thickness (um)")
    plt.ylabel("Assigned Neurite Voxel Distance to nuclei (um)")
    plt.title("DT Comparison per Sample")
    plt.legend(bbox_to_anchor=(0.5, 1), loc='upper left')

    fig_dir = os.path.join(output_path, "figures")
    os.makedirs(fig_dir, exist_ok=True)

    plt.savefig(os.path.join(fig_dir, "dt_scatterplot.png"), dpi=300)
    plt.close()

    # Scatter plot (the moment of truth)
    plt.figure(figsize=(10, 6))

    sns.scatterplot(
        x='log_dt_neurite',
        y='log_dt_nuclei',
        hue='file_short',
        data=final_df,
        palette='tab10'
    )

    plt.xlabel("Log of Assigned Neurite Voxel thickness (um)")
    plt.ylabel("Log of Assigned Neurite Voxel Distance to nuclei (um)")
    plt.title("Log DT Comparison per Sample")
    plt.legend(bbox_to_anchor=(0.5, 1), loc='upper left')

    fig_dir = os.path.join(output_path, "figures")
    os.makedirs(fig_dir, exist_ok=True)

    plt.savefig(os.path.join(fig_dir, "log_dt_scatterplot.png"), dpi=300)
    plt.close()


    print(f"Figures saved in: {fig_dir}")
    print("Pipeline complete.")

In [17]:
input_path = r"C:\Users\qfavey\Documents\Thomas-test-data\Data"
output_path="C:/Users/qfavey/Documents/Thomas-test-data/Test_Run"
run_pipeline3(input_path=input_path,output_path=output_path)

  0%|          | 0/3 [00:00<?, ?it/s]

Processing EXP_E6_d40_WTSII_DAPI_PCN488_TUJ1568_ARL13B647_40x_2025-08-19_(6).ims...
Opening .ims file: C:\Users\qfavey\Documents\Thomas-test-data\Data\EXP_E6_d40_WTSII_DAPI_PCN488_TUJ1568_ARL13B647_40x_2025-08-19_(6).ims
Opening readonly file: C:\Users\qfavey\Documents\Thomas-test-data\Data\EXP_E6_d40_WTSII_DAPI_PCN488_TUJ1568_ARL13B647_40x_2025-08-19_(6).ims 

Shape (T, C, Z, Y, X): (1, 4, 23, 2040, 2040)
Number of timepoints: 1
Number of channels: 4
Image dimensions (Z, Y, X): (23, 2040, 2040)
Z Resolution: 0.383
XY Resolution: (0.152, 0.152)


C:\Users\qfavey\AppData\Local\Temp\ipykernel_68952\2092133216.py:64: RuntimeWarning: divide by zero encountered in divide
  map_ratio = distance_map_nuclei / distance_map_neurites
C:\Users\qfavey\AppData\Local\Temp\ipykernel_68952\2092133216.py:64: RuntimeWarning: invalid value encountered in divide
  map_ratio = distance_map_nuclei / distance_map_neurites
C:\Users\qfavey\AppData\Local\Temp\ipykernel_68952\2092133216.py:127: RuntimeWarning: divide by zero encountered in log
  "log_ratio": np.log(ratio),
C:\Users\qfavey\AppData\Local\Temp\ipykernel_68952\2092133216.py:131: RuntimeWarning: divide by zero encountered in log
  "log_dt_nuclei": np.log(dt_nuclei),
C:\Users\qfavey\AppData\Local\Temp\ipykernel_68952\2092133216.py:163: RuntimeWarning: divide by zero encountered in log
  axes[0,1].imshow( np.log(map_ratio[map_ratio.shape[0] // 2]),cmap='coolwarm')
 33%|███▎      | 1/3 [00:47<01:35, 47.67s/it]

Processing EXP_E6_d40_WTSII_DAPI_PCN488_TUJ1568_ARL13B647_40x_2025-08-19_(7).ims...
Opening .ims file: C:\Users\qfavey\Documents\Thomas-test-data\Data\EXP_E6_d40_WTSII_DAPI_PCN488_TUJ1568_ARL13B647_40x_2025-08-19_(7).ims
Opening readonly file: C:\Users\qfavey\Documents\Thomas-test-data\Data\EXP_E6_d40_WTSII_DAPI_PCN488_TUJ1568_ARL13B647_40x_2025-08-19_(7).ims 

Shape (T, C, Z, Y, X): (1, 4, 23, 2040, 2040)
Number of timepoints: 1
Number of channels: 4
Image dimensions (Z, Y, X): (23, 2040, 2040)
Z Resolution: 0.383
XY Resolution: (0.152, 0.152)
Closing file: C:\Users\qfavey\Documents\Thomas-test-data\Data\EXP_E6_d40_WTSII_DAPI_PCN488_TUJ1568_ARL13B647_40x_2025-08-19_(6).ims 



C:\Users\qfavey\AppData\Local\Temp\ipykernel_68952\2092133216.py:64: RuntimeWarning: divide by zero encountered in divide
  map_ratio = distance_map_nuclei / distance_map_neurites
C:\Users\qfavey\AppData\Local\Temp\ipykernel_68952\2092133216.py:64: RuntimeWarning: invalid value encountered in divide
  map_ratio = distance_map_nuclei / distance_map_neurites
C:\Users\qfavey\AppData\Local\Temp\ipykernel_68952\2092133216.py:127: RuntimeWarning: divide by zero encountered in log
  "log_ratio": np.log(ratio),
C:\Users\qfavey\AppData\Local\Temp\ipykernel_68952\2092133216.py:131: RuntimeWarning: divide by zero encountered in log
  "log_dt_nuclei": np.log(dt_nuclei),
C:\Users\qfavey\AppData\Local\Temp\ipykernel_68952\2092133216.py:163: RuntimeWarning: divide by zero encountered in log
  axes[0,1].imshow( np.log(map_ratio[map_ratio.shape[0] // 2]),cmap='coolwarm')
 67%|██████▋   | 2/3 [01:34<00:46, 46.91s/it]

Processing EXP_E6_d40_WTSII_DAPI_PCN488_TUJ1568_ARL13B647_40x_2025-08-19_(9).ims...
Opening .ims file: C:\Users\qfavey\Documents\Thomas-test-data\Data\EXP_E6_d40_WTSII_DAPI_PCN488_TUJ1568_ARL13B647_40x_2025-08-19_(9).ims
Opening readonly file: C:\Users\qfavey\Documents\Thomas-test-data\Data\EXP_E6_d40_WTSII_DAPI_PCN488_TUJ1568_ARL13B647_40x_2025-08-19_(9).ims 

Shape (T, C, Z, Y, X): (1, 4, 23, 2040, 2040)
Number of timepoints: 1
Number of channels: 4
Image dimensions (Z, Y, X): (23, 2040, 2040)
Z Resolution: 0.383
XY Resolution: (0.152, 0.152)
Closing file: C:\Users\qfavey\Documents\Thomas-test-data\Data\EXP_E6_d40_WTSII_DAPI_PCN488_TUJ1568_ARL13B647_40x_2025-08-19_(7).ims 



C:\Users\qfavey\AppData\Local\Temp\ipykernel_68952\2092133216.py:64: RuntimeWarning: divide by zero encountered in divide
  map_ratio = distance_map_nuclei / distance_map_neurites
C:\Users\qfavey\AppData\Local\Temp\ipykernel_68952\2092133216.py:64: RuntimeWarning: invalid value encountered in divide
  map_ratio = distance_map_nuclei / distance_map_neurites
C:\Users\qfavey\AppData\Local\Temp\ipykernel_68952\2092133216.py:127: RuntimeWarning: divide by zero encountered in log
  "log_ratio": np.log(ratio),
C:\Users\qfavey\AppData\Local\Temp\ipykernel_68952\2092133216.py:131: RuntimeWarning: divide by zero encountered in log
  "log_dt_nuclei": np.log(dt_nuclei),
C:\Users\qfavey\AppData\Local\Temp\ipykernel_68952\2092133216.py:163: RuntimeWarning: divide by zero encountered in log
  axes[0,1].imshow( np.log(map_ratio[map_ratio.shape[0] // 2]),cmap='coolwarm')
100%|██████████| 3/3 [02:22<00:00, 47.36s/it]
c:\Users\qfavey\.conda\envs\napari-env-q\lib\site-packages\pandas\core\arraylike.py:399

Saved QC Excel: C:/Users/qfavey/Documents/Thomas-test-data/Test_Run\csv\all_cilia_features.xlsx
Figures saved in: C:/Users/qfavey/Documents/Thomas-test-data/Test_Run\figures
Pipeline complete.
Closing file: C:\Users\qfavey\Documents\Thomas-test-data\Data\EXP_E6_d40_WTSII_DAPI_PCN488_TUJ1568_ARL13B647_40x_2025-08-19_(9).ims 



In [ ]:
import napari

def run_pipeline2_debug_napari(
    input_path,
    output_path,
    dist_cutoff_um=2.0,
    nuclei_spot_sigma=15,
    tophat_radius=12,
    neurite_spot_sigma=5,
):
    os.makedirs(output_path, exist_ok=True)

    for file in os.listdir(input_path):
        if not file.endswith(".ims"):
            continue

        print(f"Processing {file}...")

               # -------------------------
        # Load data
        # -------------------------
        a, meta = reader.load_image(os.path.join(input_path, file))
        voxel_size = meta["voxel_size"]
        
        a_norm = preprocessing.normalize_intensity(a)
        
        # -------------------------
        # Segmentation
        # -------------------------
        cilia_labels = cilia_process_3d_ml(a[0, 0])

        nuclei_labels = nuclei_process_3d(
            a_norm[0, 3],
            tophat_radius=(tophat_radius, tophat_radius, tophat_radius),
            spot_sigma=nuclei_spot_sigma,
            outline_sigma=3,
        )

        skeleton, neurites_label = neurites_process_gpu(
            a_norm[0, 1],
            nuclei_labels,
            spot_sigma=neurite_spot_sigma,
        )

        neurite_mask = np.asarray(neurites_label) > 0
        skeleton_mask = np.asarray(skeleton) > 0

        # -------------------------
        # Distance maps
        # -------------------------
        distance_map_nuclei = distance_transform_edt(
            ~(nuclei_labels > 0), sampling=voxel_size
        )

        distance_map_neurites = distance_transform_edt(
            neurite_mask, sampling=voxel_size
        )

        dist_to_neurite, nearest_idx = distance_transform_edt(
            ~neurite_mask,
            return_indices=True,
            sampling=voxel_size,
        )

        _, nearest_skel_idx = distance_transform_edt(
            ~skeleton_mask,
            return_indices=True,
            sampling=voxel_size,
        )

        # -------------------------
        # Cilia centroids
        # -------------------------
        cilia_ids = np.unique(cilia_labels)
        cilia_ids = cilia_ids[cilia_ids != 0]

        centroids = center_of_mass(
            cilia_labels > 0, labels=cilia_labels, index=cilia_ids
        )

        centroids = np.array(centroids)

        # -------------------------
        # Napari visualization
        # -------------------------
        viewer = napari.Viewer()

        # Raw channels
        viewer.add_image(a[0, 0], name="Cilia channel", colormap="green")
        viewer.add_image(a[0, 1], name="Neurite channel", colormap="magenta")
        viewer.add_image(a[0, 3], name="Nuclei channel", colormap="blue")

        # Segmentations
        viewer.add_labels(cilia_labels, name="Cilia labels")
        viewer.add_labels(nuclei_labels, name="Nuclei labels")
        viewer.add_labels(neurites_label, name="Neurite mask")

        # Skeleton
        viewer.add_labels(skeleton_mask.astype(int), name="Skeleton")

        # Distance maps
        viewer.add_image(distance_map_nuclei, name="DT nuclei", colormap="inferno")
        viewer.add_image(distance_map_neurites, name="DT neurite (thickness)", colormap="viridis")

        # Distance to neurite
        viewer.add_image(dist_to_neurite, name="Distance to neurite", colormap="magma")

        # Centroids
        if len(centroids) > 0:
            viewer.add_points(
                centroids,
                name="Cilia centroids",
                size=5,
                face_color="red",
            )
        # -------------------------
        # Vectors: cilia → skeleton
        # -------------------------
        vectors = []

        for centroid in centroids:
            z, y, x = np.round(centroid).astype(int)

            z = np.clip(z, 0, skeleton_mask.shape[0] - 1)
            y = np.clip(y, 0, skeleton_mask.shape[1] - 1)
            x = np.clip(x, 0, skeleton_mask.shape[2] - 1)

            # nearest skeleton voxel
            sz, sy, sx = nearest_skel_idx[:, z, y, x]

            origin = np.array([centroid[0], centroid[1], centroid[2]])
            target = np.array([sz, sy, sx])

            direction = target - origin

            vectors.append([origin, direction])

        if len(vectors) > 0:
            vectors = np.array(vectors)

            viewer.add_vectors(
                vectors,
                name="Cilia → Skeleton vectors",
                edge_color="yellow",
                edge_width=1,
                length=1,  # keep real scale
            )

        # -------------------------
        # Ratio overlay (cilia / neurite)
        # -------------------------
        # avoid division by zero
        
        ratio_cilia_overlay = distance_map_nuclei / distance_map_neurites

        viewer.add_image(
            ratio_cilia_overlay,
            name="Cilia / Neurite ratio",
            colormap="magma",
            blending="additive",
        )

        print("Napari viewer opened. Inspect your life choices.")
        napari.run()

In [ ]:
run_pipeline2_debug_napari(input_path=input_path,output_path=output_path)

Processing EXP_E6_d40_WTSII_DAPI_PCN488_TUJ1568_ARL13B647_40x_2025-08-19_(6).ims...
Opening .ims file: C:\Users\qfavey\Documents\Thomas-test-data\Data\EXP_E6_d40_WTSII_DAPI_PCN488_TUJ1568_ARL13B647_40x_2025-08-19_(6).ims
Opening readonly file: C:\Users\qfavey\Documents\Thomas-test-data\Data\EXP_E6_d40_WTSII_DAPI_PCN488_TUJ1568_ARL13B647_40x_2025-08-19_(6).ims 

Shape (T, C, Z, Y, X): (1, 4, 23, 2040, 2040)
Number of timepoints: 1
Number of channels: 4
Image dimensions (Z, Y, X): (23, 2040, 2040)
Z Resolution: 0.383
XY Resolution: (0.152, 0.152)


TypeError: only length-1 arrays can be converted to Python scalars